In [1]:
import pandas as pd
import json
from datetime import datetime

# ============================================================
# STEP 2: LOAD DATA
# ============================================================

df = pd.read_csv("data/titanic.csv")

df.info()
df.head()

# ============================================================
# STEP 3: DESCRIPTIVE STATISTICS
# ============================================================

numeric_columns = df.select_dtypes(include=["int64", "float64"])

descriptive_stats = pd.DataFrame({
    "mean": numeric_columns.mean(),
    "median": numeric_columns.median(),
    "std": numeric_columns.std()
})

descriptive_stats

# ============================================================
# STEP 4: MISSING VALUES ANALYSIS
# ============================================================

missing_data = {}

for col in df.columns:
    missing_count = df[col].isna().sum()
    missing_percent = (missing_count / len(df)) * 100

    missing_data[col] = {
        "missing_count": int(missing_count),
        "missing_percent": round(missing_percent, 2)
    }

missing_df = pd.DataFrame(missing_data).T
missing_df.sort_values(by="missing_percent", ascending=False)

# ============================================================
# STEP 5: FEATURE ENGINEERING
# ============================================================

df_engineered = df.copy()

# Feature 1: FamilySize
df_engineered["FamilySize"] = (
    df_engineered["SibSp"] + df_engineered["Parch"] + 1
)

# Feature 2: IsAlone
df_engineered["IsAlone"] = df_engineered["FamilySize"].apply(
    lambda x: 1 if x == 1 else 0
)

# Feature 3: AgeGroup
def categorize_age(age):
    if pd.isna(age):
        return "Unknown"
    elif age < 18:
        return "Child"
    elif age < 30:
        return "YoungAdult"
    elif age < 50:
        return "Adult"
    else:
        return "Senior"

df_engineered["AgeGroup"] = df_engineered["Age"].apply(categorize_age)

df_engineered[["Age", "AgeGroup", "FamilySize", "IsAlone"]].head(10)

# ============================================================
# FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED
# ============================================================

df_engineered.groupby("Survived")["FamilySize"].agg(
    ["mean", "median", "std"]
)

# ============================================================
# STEP 6: CLASSES & JSON EXPORT
# ============================================================

class Passenger:
    def __init__(
        self,
        passenger_id,
        name,
        age,
        sex,
        survived,
        pclass,
        fare,
        embarked,
        family_size,
        is_alone,
        age_group
    ):
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = int(is_alone) if pd.notna(is_alone) else None
        self.age_group = str(age_group) if pd.notna(age_group) else None

    def to_dict(self):
        return self.__dict__


class TitanicDataset:
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []
        self._create_passengers()

    def _create_passengers(self):
        for _, row in self.dataframe.iterrows():
            self.passengers.append(
                Passenger(
                    row["PassengerId"],
                    row["Name"],
                    row["Age"],
                    row["Sex"],
                    row["Survived"],
                    row["Pclass"],
                    row["Fare"],
                    row["Embarked"],
                    row["FamilySize"],
                    row["IsAlone"],
                    row["AgeGroup"]
                )
            )

    def get_summary_stats(self):
        ages = [p.age for p in self.passengers if p.age is not None]
        survived = [p for p in self.passengers if p.survived == 1]

        return {
            "total_passengers": len(self.passengers),
            "survived": len(survived),
            "did_not_survive": len(self.passengers) - len(survived),
            "average_age": round(sum(ages) / len(ages), 2)
        }

    def to_json(self, filename="titanic_data.json"):
        data = {
            "metadata": {
                "dataset_name": "Titanic Passenger Dataset",
                "export_date": datetime.now().isoformat(),
                "total_passengers": len(self.passengers),
                "survival_rate": round(self.dataframe["Survived"].mean(), 4)
            },
            "summary_statistics": self.get_summary_stats(),
            "missing_values": missing_data,
            "passengers": [p.to_dict() for p in self.passengers]
        }

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        return f"JSON exported to {filename}"

# ============================================================
# STEP 7: RUN & VALIDATE JSON
# ============================================================

dataset = TitanicDataset(df_engineered)
dataset.to_json()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


'JSON exported to titanic_data.json'